# Phase 3 &mdash; Silver Layer

The Silver stage turns the faithful-but-messy Bronze table into a clean, analysis-ready dataset. Everything we do here is portable from our Phase 1 cleaning script (`src/03_data_cleaning.py`), rewritten in Spark so it can scale past single-machine memory.

### Input
`workspace.default.bronze_loans` (produced by notebook 01).

### Output
`workspace.default.silver_loans` &mdash; one row per completed loan, fewer columns, correct types, and a binary `charged_off` target.

### Operations applied (in order)
1. Filter to **completed loans only** (`Fully Paid` or `Charged Off`) and derive the `charged_off` target.
2. Drop columns with **more than 30% missing values** &mdash; too sparse to impute reliably.
3. Drop **data-leakage columns** (payment totals, recoveries, etc.) that only get populated after a loan closes and would trivially give away the outcome.
4. Drop **uninformative columns** (IDs, free-text, constant or near-constant fields).
5. **Normalize numeric strings**: strip units and percent signs from `term`, `int_rate`, `revol_util`.
6. **Parse text and dates**: map `emp_length` to an ordinal integer; convert `earliest_cr_line` and `issue_d` to dates.
7. **Null out impossible values** (e.g. negative `dti`, `revol_util > 200`) so they can be imputed in the Gold layer.

In [0]:
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-silver")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

BRONZE_TABLE = "workspace.default.bronze_loans"
SILVER_TABLE = "workspace.default.silver_loans"

In [0]:
df = spark.table(BRONZE_TABLE)
print(f"Bronze input: {df.count():,} rows x {len(df.columns)} columns")

Bronze input: 1,534,710 rows x 142 columns


## Step 1 &mdash; Filter to completed loans and build the target

Only keep loans that have actually closed. Anything still `Current`, `Late`, `In Grace Period`, etc. has no final label yet, so it can't be used for supervised training.

In [0]:
df = df.filter(F.col("loan_status").isin("Fully Paid", "Charged Off"))
df = df.withColumn("charged_off", (F.col("loan_status") == "Charged Off").cast(IntegerType()))
df = df.drop("loan_status")

total = df.count()
positives = df.filter(F.col("charged_off") == 1).count()
print(f"Completed loans: {total:,}")
print(f"  Fully Paid:   {(total - positives) / total:.1%}")
print(f"  Charged Off:  {positives / total:.1%}")

Completed loans: 1,355,678
  Fully Paid:   80.3%
  Charged Off:  19.7%


## Step 2 &mdash; Drop columns with more than 30% missing values

A column that is two-thirds blank gives the model very little to work with and is almost always an artifact of features added later in Lending Club's history. We count nulls per column in a single Spark pass and drop anything above the threshold.

In [0]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns
]).collect()[0].asDict()

high_missing = [c for c, n in null_counts.items() if n / total > 0.30]
print(f"Dropping {len(high_missing)} column(s) with more than 30% nulls.")
df = df.drop(*high_missing)
print(f"Remaining columns: {len(df.columns)}")

Dropping 49 column(s) with more than 30% nulls.
Remaining columns: 93


## Step 3 &mdash; Drop post-loan leakage columns

These fields are only populated once a loan has already closed (total payments received, recoveries, settlement information, last credit pull, etc.). Including them would give the model a trivially perfect signal, so they have to go.

In [0]:
leakage_cols = [
    "last_pymnt_amnt", "last_pymnt_d",
    "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee",
    "out_prncp", "out_prncp_inv",
    "hardship_flag",
    "recoveries", "collection_recovery_fee",
    "last_credit_pull_d",
    "last_fico_range_low", "last_fico_range_high",
    "debt_settlement_flag", "debt_settlement_flag_date",
    "settlement_status", "settlement_date",
    "settlement_amount", "settlement_percentage", "settlement_term",
]
existing = [c for c in leakage_cols if c in df.columns]
df = df.drop(*existing)
print(f"Dropped {len(existing)} leakage column(s). Remaining: {len(df.columns)}")

Dropped 16 leakage column(s). Remaining: 77


## Step 4 &mdash; Drop uninformative columns

- IDs (`id`, `member_id`, `url`) &mdash; can't generalize across rows.
- Free text (`desc`, `emp_title`, `title`) &mdash; too noisy for this project's scope.
- Near-duplicates of kept columns (`funded_amnt` vs `loan_amnt`).
- Constant or near-constant columns in 2014&ndash;2017 data (`policy_code`, `pymnt_plan`).
- `grade` &mdash; almost perfectly correlated with `int_rate`; we keep `int_rate`.

In [0]:
uninformative_cols = [
    "id", "member_id", "url",
    "desc", "title", "emp_title",
    "zip_code",
    "policy_code", "pymnt_plan",
    "funded_amnt", "funded_amnt_inv",
    "next_pymnt_d",
    "installment",
    "disbursement_method", "application_type",
    "Unnamed: 0", "unnamed_0",
    "grade",
    "tax_liens", "acc_now_delinq", "num_sats", "delinq_amnt",
    "num_tl_30dpd", "chargeoff_within_12_mths",
    "collections_12_mths_ex_med", "num_tl_120dpd_2m",
]
existing = [c for c in uninformative_cols if c in df.columns]
df = df.drop(*existing)
print(f"Dropped {len(existing)} uninformative column(s). Remaining: {len(df.columns)}")

Dropped 21 uninformative column(s). Remaining: 56


## Step 5 &mdash; Normalize numeric strings

A few columns are stored as strings with embedded units. Strip the units and cast to proper numeric types.

| Column | Before | After |
|---|---|---|
| `term`       | `" 36 months"` | `36` (int) |
| `int_rate`   | `"12.5%"`      | `12.5` (float) |
| `revol_util` | `"45.3%"`      | `45.3` (float) |

In [0]:
if "term" in df.columns:
    df = df.withColumn(
        "term",
        F.regexp_extract(F.trim(F.col("term").cast("string")), r"(\d+)", 1).cast(IntegerType()),
    )

for col in ["int_rate", "revol_util"]:
    if col in df.columns:
        df = df.withColumn(
            col,
            F.regexp_replace(F.col(col).cast("string"), "%", "").cast(DoubleType()),
        )

df.select("term", "int_rate", "revol_util").show(3)

+----+--------+----------+
|term|int_rate|revol_util|
+----+--------+----------+
|  36|    7.21|       4.5|
|  36|    5.32|      21.2|
|  36|   10.91|      53.0|
+----+--------+----------+
only showing top 3 rows


## Step 6 &mdash; Parse text and date columns

- `emp_length`: `"< 1 year"` -> 0, `"5 years"` -> 5, `"10+ years"` -> 10, missing -> -1. The -1 sentinel keeps the "missing" signal without introducing NaNs that downstream stages would need to impute.
- `earliest_cr_line`: `"Oct-1981"` -> a real `date`.
- `issue_d`: convert from string to `date` so we can filter / sort chronologically in the Gold layer.

In [0]:
if "emp_length" in df.columns:
    df = df.withColumn(
        "emp_length",
        F.when(F.col("emp_length") == "< 1 year", 0)
         .when(F.col("emp_length") == "10+ years", 10)
         .when(F.col("emp_length").isNull(), -1)
         .otherwise(F.regexp_extract(F.col("emp_length"), r"(\d+)", 1).cast(IntegerType()))
    )
    df = df.fillna({"emp_length": -1})

if "earliest_cr_line" in df.columns:
    df = df.withColumn(
        "earliest_cr_line",
        F.to_date(F.col("earliest_cr_line").cast("string"), "MMM-yyyy"),
    )

if "issue_d" in df.columns:
    df = df.withColumn("issue_d", F.to_date(F.col("issue_d").cast("string")))

df.select("emp_length", "earliest_cr_line", "issue_d").show(3)

+----------+----------------+----------+
|emp_length|earliest_cr_line|   issue_d|
+----------+----------------+----------+
|        10|      1972-10-01|2017-08-01|
|         6|      2000-12-01|2017-08-01|
|        10|      2004-01-01|2017-08-01|
+----------+----------------+----------+
only showing top 3 rows


In [0]:
numeric_fixes = {
    "annual_inc":     "double",   
    "dti":            "double",
    "open_acc":       "double",
    "revol_util":     "double",
    "fico_range_low": "int",
    "total_acc":      "int",
}
for col_name, target_type in numeric_fixes.items():
    if col_name in df.columns:
        df = df.withColumn(col_name, F.col(col_name).cast(target_type))
        print(f"  Cast '{col_name}' -> {target_type}")
print("Done.")

  Cast 'annual_inc' -> double
  Cast 'dti' -> double
  Cast 'open_acc' -> double
  Cast 'revol_util' -> double
  Cast 'fico_range_low' -> int
  Cast 'total_acc' -> int
Done.


## Step 7 &mdash; Null out impossible values

A handful of rows have values that are physically impossible (negative debt-to-income, negative annual income, revolving utilization above 200%). We set those to `NULL` so the Gold layer's median imputation handles them, rather than letting impossible values contaminate the model.

In [0]:
impossible_rules = [
    ("dti",        F.col("dti") < 0),
    ("annual_inc", F.col("annual_inc") < 0),
    ("open_acc",   F.col("open_acc") < 0),
    ("revol_util", F.col("revol_util") > 200),
]
for col, cond in impossible_rules:
    if col in df.columns:
        df = df.withColumn(col, F.when(cond, None).otherwise(F.col(col)))
print("Applied domain constraints.")

Applied domain constraints.


## Step 8 &mdash; Write the Silver Delta table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

final = spark.table(SILVER_TABLE)
print(f"Wrote Delta table: {SILVER_TABLE}")
print(f"Final shape: {final.count():,} rows x {len(final.columns)} columns")

Wrote Delta table: workspace.default.silver_loans
Final shape: 1,355,678 rows x 56 columns


In [0]:
spark.table(SILVER_TABLE).limit(5).toPandas()

,loan_amnt,term,int_rate,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_inq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,charged_off
0,8000,36,7.21,A3,10,RENT,44000.0,Not Verified,2017-08-01,car,CA,17.81,0.0,1972-10-01,685,689,1.0,6.0,1,461,4.5,21,w,0.0,3611.0,10200.0,2.0,722.0,1941.0,11.8,74.0,538.0,21.0,21.0,1.0,91.0,1.0,0.0,1.0,2.0,1.0,7.0,3.0,4.0,17.0,2.0,0.0,0.0,100.0,0.0,1.0,29595.0,3611.0,2200.0,19395.0,0
1,11500,36,5.32,A1,6,OWN,51000.0,Not Verified,2017-08-01,credit_card,PA,20.75,0.0,2000-12-01,780,784,0.0,14.0,0,11406,21.2,28,w,0.0,43100.0,53900.0,3.0,3079.0,34360.0,24.8,119.0,200.0,9.0,9.0,2.0,9.0,9.0,0.0,7.0,8.0,9.0,11.0,5.0,13.0,21.0,8.0,0.0,1.0,100.0,0.0,0.0,92843.0,43100.0,45700.0,38943.0,0
2,15000,36,10.91,B4,10,RENT,60000.0,Verified,2017-08-01,debt_consolidation,IL,28.26,2.0,2004-01-01,670,674,0.0,10.0,0,11976,53.0,21,f,0.0,43210.0,22600.0,4.0,4801.0,6798.0,57.5,162.0,117.0,19.0,14.0,0.0,21.0,14.0,0.0,2.0,5.0,3.0,3.0,9.0,8.0,12.0,5.0,1.0,0.0,85.7,33.3,0.0,83197.0,43210.0,16000.0,60597.0,0
3,7500,36,7.07,A2,2,MORTGAGE,40000.0,Source Verified,2017-08-01,debt_consolidation,WA,8.25,0.0,2005-11-01,785,789,0.0,9.0,0,4869,10.7,18,w,0.0,7468.0,45400.0,3.0,830.0,20846.0,15.3,43.0,141.0,12.0,12.0,2.0,18.0,5.0,0.0,1.0,2.0,4.0,6.0,1.0,8.0,15.0,2.0,0.0,1.0,100.0,0.0,0.0,52400.0,7468.0,24600.0,7000.0,0
4,18000,36,7.97,A5,10,MORTGAGE,100000.0,Verified,2017-08-01,debt_consolidation,MA,22.81,0.0,2003-05-01,700,704,1.0,8.0,0,168484,95.0,14,w,0.0,436843.0,176700.0,4.0,54605.0,7140.0,77.0,154.0,140.0,13.0,6.0,4.0,13.0,4.0,0.0,5.0,6.0,6.0,7.0,2.0,6.0,10.0,5.0,0.0,0.0,93.0,60.0,0.0,494551.0,53826.0,35700.0,34901.0,0
